# Imports

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()

In [ ]:
from sklearn.datasets import load_breast_cancer, fetch_covtype
from sklearn.tree import DecisionTreeClassifier

from tools.tree_reader.decision_path_extractor import (
    DecisionPathExtractor
)

from tools.tree_reader.feature_importance_extractor import (
    FeatureImportanceExtractor
)

from tools.tree_reader.tree_structure_exporter import (
    TreeStructureExporter
)

# Train Model

In [ ]:
data = load_breast_cancer()

X = data.data
y = data.target

model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

model.fit(X, y)

# Simulate User Question

In [ ]:
user_question = (
    "Why was record 0 classified?"
)

record_id = 0

sample = X[record_id]

# Planner Agent

In [ ]:
def planner(question):

    return [
        "decision_path",
        "feature_importance",
        "prediction"
    ]

# Decision Path Tool

In [ ]:
extractor = (
    DecisionPathExtractor(
        model,
        data.feature_names
    )
)

decision_path = (
    extractor.extract_path(
        sample
    )
)

decision_path

# Feature Importance Tool

In [ ]:
importance = (
    FeatureImportanceExtractor(
        model,
        data.feature_names
    )
)

importance_df = (
    importance.get_top_features(5)
)

importance_df

# Prediction Tool

In [ ]:
prediction = model.predict(
    sample.reshape(1, -1)
)[0]

prediction_name = (
    data.target_names[
        prediction
    ]
)

prediction_name

# LLM Explanation

In [ ]:
context = {
    "prediction": prediction_name,
    "decision_path": decision_path,
    "top_features": (
        importance_df
        .to_dict(
            orient="records"
        )
    )
}

context

In [ ]:
print(os.getenv("SSL_CERT_FILE"))

In [ ]:
os.environ.pop("SSL_CERT_FILE", None)

In [ ]:
print(os.getenv("SSL_CERT_FILE"))

In [ ]:
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [ ]:
prompt = f"""
You are an Explainable AI expert.

Prediction:
{context['prediction']}

Decision Path:
{context['decision_path']}

Important Features:
{context['top_features']}

Explain in business-friendly language.
"""

In [ ]:
response = client.responses.create(
        model="gpt-4.1-mini",
        input= prompt
    )

In [ ]:
print("\n===== AGENT EXPLANATION =====\n")
print(response.output_text)